# MarketMinds — Phase 3c: Sequence Model (LSTM/GRU)

Phase 3/3b found that four different modeling approaches (linear regression, ARIMA, Random Forest, XGBoost) all show the same shape of failure: some real skill in calm markets, collapsing into significant anti-skill during COVID (worst for Random Forest, 50% of assets significantly wrong). This notebook asks the same question of a sequence model: does giving the model an explicit window of recent history (rather than a single day's snapshot of indicators) find anything the others missed, or fail the same way?

Kept identical to every prior model for comparability:
- **Same features** (`src.forecasting.FEATURE_COLS`), reused from Phase 1, nothing re-derived.
- **Same target**: 5-trading-day-forward log return (`target_log_return_5d`), no lookahead.
- **Same three windows**: `normal`, `2008_financial_crisis`, `covid_crash`.
- **Same basket** and skip logic for insufficient history.
- **Same significance test** against 50% random chance, split into significantly-better vs. significantly-worse.

**What's different, and why:**
- **Input shape**: instead of one day's feature vector, the model sees the last 30 trading days of features (a sliding window) and predicts the same 5-day-forward return from the last day in that window.
- **Training cadence**: trained *once* per (asset, window) on the expanding window up to `test_start`, then used to predict every day in that test window without refitting. Retraining a neural net at every test day (or even every 20 days, like Phase 3b's tree models) would be far too slow across ~21 assets x 3 windows. No-lookahead still holds for every individual prediction -- it's just not refreshed day-by-day within the window. This is a bigger fidelity trade-off than Phase 3b's, and it's disclosed here rather than hidden.
- **Modest architecture, deliberately**: 1-2 LSTM layers, 32 hidden units. Each window only has a few thousand training sequences and 8 features -- a large/deep network would overfit noisy daily return data long before it found anything a smaller network couldn't. This is still meant to be a baseline sequence model, not a tuned production one.
- **Device**: Apple Silicon MPS backend, no CUDA anywhere --
  ```python
  device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
  ```
  One known risk worth flagging up front: PyTorch's MPS support for `nn.LSTM` has had gaps in some versions. If you hit an MPS-specific error mentioning LSTM, paste it back -- that's a version issue to work around, not something wrong with the approach.

**Runtime note**: 21 assets x 3 windows = ~63 total training runs, each with early stopping (so actual epoch counts vary). Step 8's per-asset timing prints will show whether MPS is actually being used effectively -- if a fit is taking many seconds per epoch, something's off and worth pasting back before letting the full loop run.

**Note on execution**: same as always — you run every cell, I don't execute anything. Paste back any errors.

## Setup

In [1]:
import json
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import torch

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
torch.manual_seed(42)

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.append(str(PROJECT_ROOT))

from src.forecasting import FEATURE_COLS, build_feature_target_table, compute_eval_metrics, safe_asset_name
from src.sequence_model import build_sequences, train_eval_lstm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print('Using device:', device)

master = pq.read_table(PROCESSED_DIR / 'marketminds_master.parquet').to_pandas()
symbol_metadata = pq.read_table(PROCESSED_DIR / 'symbol_metadata.parquet').to_pandas()
print('Master shape:', master.shape)
print('Features (reused from Phase 1, unchanged):', FEATURE_COLS)

Using device: mps
Master shape: (134646, 18)
Features (reused from Phase 1, unchanged): ['daily_return', 'ma_20', 'ma_50', 'ma_200', 'rsi_14', 'macd', 'macd_signal', 'volatility_20d']


## 1. Reuse Phase 3's windows and basket definitions

Identical to Phase 3/3b. `MIN_TRAIN_SEQUENCES` accounts for the fact that building sequences consumes the first `SEQ_LEN - 1` rows as pure history before the first usable training example.

In [2]:
ASSETS = sorted(t for t in master['ticker'].unique() if t != '^INDIAVIX')

WINDOWS = {
    'normal': {'train_end': '2016-12-31', 'test_start': '2017-01-01', 'test_end': '2017-12-31'},
    '2008_financial_crisis': {'train_end': '2007-12-31', 'test_start': '2008-01-01', 'test_end': '2009-03-31'},
    'covid_crash': {'train_end': '2019-12-31', 'test_start': '2020-02-20', 'test_end': '2020-04-30'},
}

MIN_TRAIN_SEQUENCES = 250

print(len(ASSETS), 'assets:', ASSETS)

20 assets: ['DRREDDY.NS', 'GOLDBEES.NS', 'HDFCBANK.NS', 'HINDUNILVR.NS', 'ICICIBANK.NS', 'INFY.NS', 'ITC.NS', 'KOTAKBANK.NS', 'LT.NS', 'MARUTI.NS', 'NESTLEIND.NS', 'ONGC.NS', 'RELIANCE.NS', 'SUNPHARMA.NS', 'TATASTEEL.NS', 'TCS.NS', 'ULTRACEMCO.NS', 'WIPRO.NS', '^BSESN', '^NSEI']


## 2. Sequence design and architecture

`SEQ_LEN = 30` trading days (~6 weeks) of history per sequence -- long enough to give the LSTM something to find beyond a single day's snapshot, short enough to keep the per-window training set (a few thousand sequences) comfortably larger than the parameter count of a 32-unit, 1-layer LSTM. `cell_type` can be switched to `'gru'` with no other changes, using the same `LSTMForecaster` class (it dispatches to `nn.GRU` internally).

In [3]:
SEQ_LEN = 30
HIDDEN_SIZE = 32
NUM_LAYERS = 1
CELL_TYPE = 'lstm'

# Sanity check: confirm the sequence shape looks right before running the full loop.
sample_ticker = 'RELIANCE.NS'
sample_table = build_feature_target_table(master[master['ticker'] == sample_ticker])
X_sample, y_sample, dates_sample = build_sequences(sample_table, SEQ_LEN)
print(sample_ticker, '-- X shape:', X_sample.shape, '(n_sequences, seq_len, n_features), y shape:', y_sample.shape)
print('First sequence date:', dates_sample[0], ', last:', dates_sample[-1])

RELIANCE.NS -- X shape: (6709, 30, 8) (n_sequences, seq_len, n_features), y shape: (6709,)
First sequence date: 2000-11-16 00:00:00 , last: 2026-08-04 00:00:00


## 3. Train, evaluate, and save the LSTM across the full basket

For each asset: build sequences once, then for each window, train once (early-stopped on a chronological validation split) and evaluate across the whole test period. Saves the model bundle (model + scaler + architecture config, everything needed for inference) as `.pt`, plus the same metadata convention as every prior model, extended with `seq_len`, `hidden_size`, `training_time_seconds`, and `epochs_trained` so it's visible whether MPS is actually helping.

In [4]:
def save_model(model_bundle, ticker, window_name, cfg, metrics, info):
    asset_name = safe_asset_name(ticker)
    base = 'lstm_' + asset_name + '_' + window_name
    model_path = MODELS_DIR / (base + '.pt')
    meta_path = MODELS_DIR / (base + '.json')

    torch.save(model_bundle, model_path)

    metadata = {
        'asset': ticker,
        'model_type': 'lstm',
        'window': window_name,
        'train_end': cfg['train_end'],
        'test_start': cfg['test_start'],
        'test_end': cfg['test_end'],
        'seq_len': model_bundle['seq_len'],
        'hidden_size': model_bundle['hidden_size'],
        'num_layers': model_bundle['num_layers'],
        'cell_type': model_bundle['cell_type'],
        'training_time_seconds': info['training_time_seconds'],
        'epochs_trained': info['epochs_trained'],
        'best_val_loss': info['best_val_loss'],
        'metrics': metrics,
        'date_trained': datetime.now().isoformat(),
    }
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

In [5]:
# PyTorch's MPS backend isn't fully deterministic even with a fixed seed -- re-running the same
# window can land on a meaningfully different trained model (discovered while extending this to
# more windows in 07_lstm_extended_windows.ipynb: postcovid flipped between "not significant" and
# "significantly worse" across two identical runs). Training N_REPEATS independently-seeded models
# per (asset, window) and averaging their predictions before computing metrics is standard ensemble
# variance-reduction, and directly fixes this. Applied here too so these 3 windows are held to the
# same standard as the 5 newer ones before anything gets published together.
N_REPEATS = 5

results = []
start_time = time.time()

for i, ticker in enumerate(ASSETS, 1):
    asset_df = master[master['ticker'] == ticker]
    feature_table = build_feature_target_table(asset_df)
    X, y, seq_dates = build_sequences(feature_table, SEQ_LEN)
    print(f'[{i}/{len(ASSETS)}] {ticker} -- {len(X)} sequences total, elapsed {time.time() - start_time:.0f}s')

    for window_name, cfg in WINDOWS.items():
        train_count = (seq_dates <= pd.Timestamp(cfg['train_end'])).sum()
        test_count = ((seq_dates >= pd.Timestamp(cfg['test_start'])) & (seq_dates <= pd.Timestamp(cfg['test_end']))).sum()

        if train_count < MIN_TRAIN_SEQUENCES or test_count == 0:
            results.append({
                'ticker': ticker, 'window': window_name, 'model': 'lstm',
                'status': 'skipped_insufficient_history',
                'mae_pct': np.nan, 'rmse_pct': np.nan, 'directional_accuracy_pct': np.nan,
                'dir_acc_ci_low': np.nan, 'dir_acc_ci_high': np.nan,
                'p_value_vs_random': np.nan, 'significant_vs_random': False,
                'significant_better_than_random': False, 'significant_worse_than_random': False,
                'n_predictions': int(train_count),
            })
            continue

        window_start = time.time()
        pred_runs = []
        actual, model_bundle, info = None, None, None
        for repeat in range(N_REPEATS):
            torch.manual_seed(42 + repeat)
            predicted_r, actual_r, model_bundle_r, info_r = train_eval_lstm(
                X, y, seq_dates, cfg, device,
                hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, cell_type=CELL_TYPE,
            )
            pred_runs.append(predicted_r)
            actual = actual_r
            model_bundle, info = model_bundle_r, info_r  # keep the last repeat's model as the saved artifact
        predicted = sum(pred_runs) / len(pred_runs)
        window_time = time.time() - window_start

        metrics = compute_eval_metrics(actual, predicted)
        results.append({'ticker': ticker, 'window': window_name, 'model': 'lstm', 'status': 'ok', **metrics})
        save_model(model_bundle, ticker, window_name, cfg, metrics, info)

        print(f'    {window_name}: {window_time:.1f}s for {N_REPEATS} runs, '
              f'dir_acc={metrics["directional_accuracy_pct"]}%, n={metrics["n_predictions"]}')

print(f'Done in {time.time() - start_time:.0f}s -- {len(results)} rows')
lstm_results_df = pd.DataFrame(results)

[1/20] DRREDDY.NS -- 6709 sequences total, elapsed 0s
    normal: 5.6s for 5 runs, dir_acc=46.9%, n=260
    2008_financial_crisis: 2.5s for 5 runs, dir_acc=51.5%, n=326
    covid_crash: 4.8s for 5 runs, dir_acc=60.8%, n=51
[2/20] GOLDBEES.NS -- 4360 sequences total, elapsed 13s
    normal: 2.5s for 5 runs, dir_acc=48.8%, n=260
    covid_crash: 8.4s for 5 runs, dir_acc=41.2%, n=51
[3/20] HDFCBANK.NS -- 6709 sequences total, elapsed 24s
    normal: 6.3s for 5 runs, dir_acc=40.0%, n=260
    2008_financial_crisis: 3.1s for 5 runs, dir_acc=50.0%, n=326
    covid_crash: 5.3s for 5 runs, dir_acc=60.8%, n=51
[4/20] HINDUNILVR.NS -- 6709 sequences total, elapsed 39s
    normal: 5.7s for 5 runs, dir_acc=41.5%, n=260
    2008_financial_crisis: 3.1s for 5 runs, dir_acc=49.7%, n=326
    covid_crash: 5.0s for 5 runs, dir_acc=68.6%, n=51
[5/20] ICICIBANK.NS -- 6059 sequences total, elapsed 53s
    normal: 4.0s for 5 runs, dir_acc=51.2%, n=260
    2008_financial_crisis: 1.3s for 5 runs, dir_acc=53.7%,

## 4. Merge sector metadata, save detailed LSTM results

In [6]:
lstm_results_df = lstm_results_df.merge(symbol_metadata, on='ticker', how='left')
cols = ['ticker', 'category', 'window', 'model', 'status', 'mae_pct', 'rmse_pct',
        'directional_accuracy_pct', 'dir_acc_ci_low', 'dir_acc_ci_high',
        'p_value_vs_random', 'significant_vs_random',
        'significant_better_than_random', 'significant_worse_than_random', 'n_predictions']
lstm_results_df = lstm_results_df[cols]

lstm_path = RESULTS_DIR / 'lstm_model_validation_full_basket.csv'
lstm_results_df.to_csv(lstm_path, index=False)
print('Saved', len(lstm_results_df), 'rows to', lstm_path)
lstm_results_df.head(10)

Saved 60 rows to /Users/palakjagtap/Documents/Market_minds/results/lstm_model_validation_full_basket.csv


,ticker,category,window,model,status,mae_pct,rmse_pct,directional_accuracy_pct,dir_acc_ci_low,dir_acc_ci_high,p_value_vs_random,significant_vs_random,significant_better_than_random,significant_worse_than_random,n_predictions
0,DRREDDY.NS,Pharma,normal,lstm,ok,3.432,4.519,46.9,40.9,53.0,0.3211,False,False,False,260
1,DRREDDY.NS,Pharma,2008_financial_crisis,lstm,ok,4.428,5.781,51.5,46.1,56.9,0.5797,False,False,False,326
2,DRREDDY.NS,Pharma,covid_crash,lstm,ok,6.859,8.288,60.8,47.1,73.0,0.1235,False,False,False,51
3,GOLDBEES.NS,Gold,normal,lstm,ok,1.144,1.402,48.8,42.8,54.9,0.7098,False,False,False,260
4,GOLDBEES.NS,Gold,2008_financial_crisis,lstm,skipped_insufficient_history,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0
5,GOLDBEES.NS,Gold,covid_crash,lstm,ok,7.656,8.621,41.2,28.8,54.8,0.2076,False,False,False,51
6,HDFCBANK.NS,Financials,normal,lstm,ok,2.545,3.179,40.0,34.2,46.1,0.0013,True,False,True,260
7,HDFCBANK.NS,Financials,2008_financial_crisis,lstm,ok,6.431,8.051,50.0,44.6,55.4,1.0000,False,False,False,326
8,HDFCBANK.NS,Financials,covid_crash,lstm,ok,6.139,8.440,60.8,47.1,73.0,0.1235,False,False,False,51
9,HINDUNILVR.NS,FMCG,normal,lstm,ok,3.893,4.667,41.5,35.7,47.6,0.0064,True,False,True,260


## 5. Five-model comparison (main output)

Combines Phase 3's `baseline_model_validation_full_basket.csv` (linear regression, ARIMA), Phase 3b's `nonlinear_model_validation_full_basket.csv` (Random Forest, XGBoost), and this notebook's LSTM results into one significance-summary table, same format as `phase3b_nonlinear_comparison.csv` -- now with all five models side by side.

In [7]:
baseline_results_df = pd.read_csv(RESULTS_DIR / 'baseline_model_validation_full_basket.csv')
nonlinear_results_df = pd.read_csv(RESULTS_DIR / 'nonlinear_model_validation_full_basket.csv')

combined_df = pd.concat([baseline_results_df, nonlinear_results_df, lstm_results_df], ignore_index=True)
print('Combined shape:', combined_df.shape)
print('Models included:', sorted(combined_df['model'].unique()))

ok_df = combined_df[combined_df['status'] == 'ok']
print()
print('Mean MAE / RMSE / directional accuracy per model per window:')
ok_df.groupby(['model', 'window'])[['mae_pct', 'rmse_pct', 'directional_accuracy_pct']].mean().round(3)

Combined shape: (300, 16)
Models included: ['arima', 'linear_regression', 'lstm', 'random_forest', 'xgboost']

Mean MAE / RMSE / directional accuracy per model per window:


mae_pct  rmse_pct  \
model             window                                     
arima             2008_financial_crisis    5.652     7.227   
                  covid_crash              7.009     8.878   
                  normal                   2.090     2.792   
linear_regression 2008_financial_crisis    5.559     7.150   
                  covid_crash              7.655     9.600   
                  normal                   2.130     2.839   
lstm              2008_financial_crisis    6.022     7.672   
                  covid_crash              7.351     9.274   
                  normal                   2.599     3.333   
random_forest     2008_financial_crisis    5.728     7.322   
                  covid_crash              8.389    10.251   
                  normal                   2.164     2.882   
xgboost           2008_financial_crisis    5.928     7.544   
                  covid_crash              8.564    11.505   
                  normal                   2.389     3.144   

                                         directional_accuracy_pct  
model             window                                           
arima             2008_financial_crisis                    45.939  
                  covid_crash                              41.270  
                  normal                                   55.375  
linear_regression 2008_financial_crisis                    54.583  
                  covid_crash                              50.795  
                  normal                                   52.590  
lstm              2008_financial_crisis                    47.844  
                  covid_crash                              51.180  
                  normal                                   48.010  
random_forest     2008_financial_crisis                    51.156  
                  covid_crash                              37.950  
                  normal                                   52.790  
xgboost           2008_financial_crisis                    53.244  
                  covid_crash                              42.555  
                  normal                                   51.075

In [8]:
comparison = ok_df.groupby(['model', 'window']).agg(
    total_assets=('ticker', 'count'),
    significant_better=('significant_better_than_random', 'sum'),
    significant_worse=('significant_worse_than_random', 'sum'),
).reset_index()
comparison['pct_significant_better'] = (comparison['significant_better'] / comparison['total_assets'] * 100).round(1)
comparison['pct_significant_worse'] = (comparison['significant_worse'] / comparison['total_assets'] * 100).round(1)

comparison_path = RESULTS_DIR / 'phase3c_five_model_comparison.csv'
comparison.to_csv(comparison_path, index=False)
print('Saved to', comparison_path)
comparison

Saved to /Users/palakjagtap/Documents/Market_minds/results/phase3c_five_model_comparison.csv


,model,window,total_assets,significant_better,significant_worse,pct_significant_better,pct_significant_worse
0,arima,2008_financial_crisis,18,1,4,5.6,22.2
1,arima,covid_crash,20,0,7,0.0,35.0
2,arima,normal,20,8,0,40.0,0.0
3,linear_regression,2008_financial_crisis,18,5,0,27.8,0.0
4,linear_regression,covid_crash,20,0,0,0.0,0.0
5,linear_regression,normal,20,6,1,30.0,5.0
6,lstm,2008_financial_crisis,18,2,1,11.1,5.6
7,lstm,covid_crash,20,2,0,10.0,0.0
8,lstm,normal,20,3,6,15.0,30.0
9,random_forest,2008_financial_crisis,18,3,4,16.7,22.2


## 6. Format for `assets/js/data.js`

Same idea as `07_lstm_extended_windows.ipynb`'s Step 4. Note the key mapping: our internal window names (`normal`, `2008_financial_crisis`, `covid_crash`) don't match her `data.js` keys for the same windows (`normal`, `gfc2008`, `covid2020`) -- handled below.

In [9]:
JS_WINDOW_KEY = {
    'normal': 'normal',
    '2008_financial_crisis': 'gfc2008',
    'covid_crash': 'covid2020',
}

def js_bool(v):
    return 'true' if v else 'false'

print('// --- paste into MM.ML_METRICS.bsesn.<window>.lstm for each window ---\n')
for window_name, js_key in JS_WINDOW_KEY.items():
    row = lstm_results_df[(lstm_results_df['ticker'] == '^BSESN') & (lstm_results_df['window'] == window_name)]
    if row.empty or row.iloc[0]['status'] != 'ok':
        print(f'// {js_key}: no SENSEX result (skipped or missing)')
        continue
    r = row.iloc[0]
    ci = f"[{r['dir_acc_ci_low']}, {r['dir_acc_ci_high']}]" if pd.notna(r['dir_acc_ci_low']) else 'null'
    print(f"// {js_key}")
    print(f"lstm: {{ mae_pct: {r['mae_pct']}, rmse_pct: {r['rmse_pct']}, dir_acc_pct: {r['directional_accuracy_pct']}, "
          f"ci: {ci}, sig_better: {js_bool(r['significant_better_than_random'])}, "
          f"sig_worse: {js_bool(r['significant_worse_than_random'])}, p: {r['p_value_vs_random']}, n: {int(r['n_predictions'])} }},\n")

print()
print('// --- append to MM.ML_METRICS.basketSignificance ---\n')
for window_name, js_key in JS_WINDOW_KEY.items():
    ok_rows = lstm_results_df[(lstm_results_df['window'] == window_name) & (lstm_results_df['status'] == 'ok')]
    total = len(ok_rows)
    if total == 0:
        print(f'// {js_key}: no results')
        continue
    sig_better = int(ok_rows['significant_better_than_random'].sum())
    sig_worse = int(ok_rows['significant_worse_than_random'].sum())
    pct_better = round(sig_better / total * 100, 1)
    pct_worse = round(sig_worse / total * 100, 1)
    print(f"{{ model: 'lstm', window: '{js_key}', total: {total}, sigBetter: {sig_better}, sigWorse: {sig_worse}, pctBetter: {pct_better}, pctWorse: {pct_worse} }},")

// --- paste into MM.ML_METRICS.bsesn.<window>.lstm for each window ---

// normal
lstm: { mae_pct: 1.082, rmse_pct: 1.336, dir_acc_pct: 51.5, ci: [45.5, 57.5], sig_better: false, sig_worse: false, p: 0.6198, n: 260 },

// gfc2008
lstm: { mae_pct: 5.792, rmse_pct: 7.385, dir_acc_pct: 47.2, ci: [41.9, 52.7], sig_better: false, sig_worse: false, p: 0.3188, n: 326 },

// covid2020
lstm: { mae_pct: 6.448, rmse_pct: 8.382, dir_acc_pct: 41.2, ci: [28.8, 54.8], sig_better: false, sig_worse: false, p: 0.2076, n: 51 },


// --- append to MM.ML_METRICS.basketSignificance ---

{ model: 'lstm', window: 'normal', total: 20, sigBetter: 3, sigWorse: 6, pctBetter: 15.0, pctWorse: 30.0 },
{ model: 'lstm', window: 'gfc2008', total: 18, sigBetter: 2, sigWorse: 1, pctBetter: 11.1, pctWorse: 5.6 },
{ model: 'lstm', window: 'covid2020', total: 20, sigBetter: 2, sigWorse: 0, pctBetter: 10.0, pctWorse: 0.0 },


## Summary

What this notebook produced:
- `results/lstm_model_validation_full_basket.csv` -- detailed per-asset LSTM results, same format as Phase 3/3b's files
- `results/phase3c_five_model_comparison.csv` -- all five models (linear regression, ARIMA, Random Forest, XGBoost, LSTM) x three windows, significance-summary format
- Trained LSTM models in `models/` (one `.pt` + `.json` per asset/window)

The question this was built to answer: **does giving the model an explicit window of recent history find signal the single-day-snapshot models missed, or does it fail the same way during COVID?** Compare the `lstm` row's `pct_significant_better`/`pct_significant_worse` in Step 5's table against the other four models across the three windows.

Next: depending on what Step 5 shows, either treat five converging results as the finished, honest ML-evaluation story for the write-up, or investigate further (e.g. a GRU variant, a longer sequence window) -- and either way, the Next.js frontend / app integration is the remaining major piece per the roadmap.